# RSNA Knee V38: five-branch rank fusion

This post-competition notebook mirrors the independently written fusion code in [https://github.com/kaishikan750-cyber/rsna-knee-silver-solution](https://github.com/kaishikan750-cyber/rsna-knee-silver-solution).

The author's reported competition result is **Solo silver, 311 / 6,300 (top 4.94%), private ROC-AUC 0.943**.
The archived V38 recipe matches the five-branch description; saved historical records show public 0.958 in October 2026. The private score 0.943 is separately reported by the participant. This synthetic demo does not independently establish that score or the final notebook selection.

## Scope

Input: predictions from Raptor384, Raptor224, R6 mirror-TTA, S6 and the three-fold 2.5D Reader.
This notebook demonstrates post-processing with **synthetic cases only**. It contains no MRI images, clinical reports, labels, model checkpoints or actual case predictions. It does not retrain the upstream models or rerun MRI inference, and it is not a competition submission.

Each finding is ranked across the complete case set with average ties. Both Raptor384/Raptor224 and R6/S6 use 65/35 internal weights. Per-finding group weights are followed by a small Reader contribution and final re-ranking. The archived float32 CSV round-trip is retained.

Upstream methods and assets: [goodpjw2008](https://www.kaggle.com/code/goodpjw2008/rsna-knee-coatnet-r6-2-5d-mil-lb-0-958), [nartaa](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror), [heliosli](https://www.kaggle.com/code/heliosli/this-is-new-version-0-9576-in-one-click), and [varshithakolupuri](https://www.kaggle.com/code/varshithakolupuri/rsna-knee-one-unused-checkpoint-lifts-the-0-956). Full attribution and the role of each public model are in the repository's `docs/PROVENANCE.md`; training these upstream models is not claimed as the author's original work.


In [ ]:
"""Per-finding percentile-rank fusion; no model weights or patient data included."""

from __future__ import annotations

import io
import json
from pathlib import Path
from typing import Mapping

import numpy as np
import pandas as pd

STAGES = ("raptor384", "raptor224", "r6", "s6", "reader")
ID_COLUMN = "StudyInstanceUID"


def load_config(path: str | Path) -> dict:
    config = json.loads(Path(path).read_text(encoding="utf-8"))
    validate_config(config)
    return config


def validate_config(config: Mapping) -> None:
    labels = config["target_order"]
    if not isinstance(labels, list) or len(labels) != 12 or len(set(labels)) != 12:
        raise ValueError("target_order must contain 12 distinct finding names")
    for field in ("raptor_weights", "reader_weights"):
        if set(config[field]) != set(labels):
            raise ValueError(f"{field} must specify exactly the configured findings")
        values = np.array([config[field][label] for label in labels], dtype=float)
        if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
            raise ValueError(f"{field} must be finite and within [0, 1]")
    for field in ("raptor384_share", "s6_share"):
        value = config[field]
        if isinstance(value, bool) or not np.isfinite(value) or not 0 <= value <= 1:
            raise ValueError(f"{field} must be within [0, 1]")
    if not isinstance(config.get("base_float32_csv_roundtrip"), bool):
        raise ValueError("base_float32_csv_roundtrip must be explicit true or false")


def percentile_rank(scores: np.ndarray) -> np.ndarray:
    """Rank each finding across cases, using average ties and percentiles 1/N..1."""
    values = np.asarray(scores, dtype=np.float64)
    if values.ndim != 2 or values.shape[0] < 2 or values.shape[1] == 0:
        raise ValueError("scores must be a 2D array containing at least two cases")
    if not np.isfinite(values).all():
        raise ValueError("scores must be finite")
    return pd.DataFrame(values).rank(method="average", pct=True).to_numpy()


def blend_scores(stages: Mapping[str, np.ndarray], config: Mapping) -> np.ndarray:
    """Return the V38 final rank scores for five already-TTA-aggregated branches.

    The float32 -> CSV -> float64 step intentionally follows the saved notebook:
    it can affect near-ties and must precede ranking the group-weighted base.
    """
    validate_config(config)
    if set(stages) != set(STAGES):
        raise ValueError(f"exactly five branches are required: {STAGES}")
    arrays = {name: np.asarray(stages[name], dtype=np.float64) for name in STAGES}
    shape = arrays[STAGES[0]].shape
    if len(shape) != 2 or shape[0] < 2 or shape[1] != len(config["target_order"]):
        raise ValueError("branch arrays must have shape (N >= 2, 12)")
    for name, values in arrays.items():
        if values.shape != shape:
            raise ValueError(f"shape mismatch in {name}")
        if not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():
            raise ValueError(f"invalid probabilities in {name}; expected finite [0, 1]")

    ranked = {name: percentile_rank(values) for name, values in arrays.items()}
    a, b = config["raptor384_share"], config["s6_share"]
    raptor = a * ranked["raptor384"] + (1.0 - a) * ranked["raptor224"]
    compact = (1.0 - b) * ranked["r6"] + b * ranked["s6"]
    labels = config["target_order"]
    weights = np.array([config["raptor_weights"][label] for label in labels])
    reader_share = np.array([config["reader_weights"][label] for label in labels])
    base = weights * raptor + (1.0 - weights) * compact
    if config["base_float32_csv_roundtrip"]:
        buffer = io.StringIO(pd.DataFrame(base.astype(np.float32)).to_csv(index=False))
        base = pd.read_csv(buffer).to_numpy()
    return percentile_rank(
        (1.0 - reader_share) * percentile_rank(base) + reader_share * ranked["reader"]
    )


def combine_stage_frames(stages: Mapping[str, pd.DataFrame], config: Mapping) -> pd.DataFrame:
    """Align unique study IDs to the first branch, then apply column-wise fusion."""
    validate_config(config)
    if set(stages) != set(STAGES):
        raise ValueError(f"exactly five branches are required: {STAGES}")
    labels = config["target_order"]
    indexed = {}
    for name in STAGES:
        frame = stages[name]
        if frame.columns.duplicated().any() or set(frame.columns) != {ID_COLUMN, *labels}:
            raise ValueError(f"{name} must contain the ID column and exactly 12 findings")
        if frame[ID_COLUMN].isna().any():
            raise ValueError(f"missing study ID in {name}")
        ids = frame[ID_COLUMN].astype(str)
        if ids.str.strip().eq("").any() or ids.duplicated().any():
            raise ValueError(f"empty or duplicate study ID in {name}")
        indexed[name] = frame.assign(**{ID_COLUMN: ids}).set_index(ID_COLUMN)
    reference = indexed[STAGES[0]].index
    for name in STAGES[1:]:
        if set(indexed[name].index) != set(reference):
            raise ValueError(f"different study IDs in {name}")
    arrays = {name: frame.loc[reference, labels].to_numpy(dtype=np.float64)
              for name, frame in indexed.items()}
    result = pd.DataFrame(blend_scores(arrays, config), columns=labels)
    result.insert(0, ID_COLUMN, reference.to_numpy())
    return result


## Archived V38 configuration

Weights follow the saved V38 recipe; no test labels are loaded or fitted here.


In [ ]:
CONFIG = json.loads(r'''{
  "schema_version": 1,
  "recipe": "V38",
  "target_order": [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
  ],
  "raptor384_share": 0.65,
  "s6_share": 0.35,
  "raptor_weights": {
    "ACL": 0.35000000000000003,
    "MCL": 0.375,
    "Medial Meniscus": 0.5,
    "Lateral Meniscus": 0.4,
    "Medial OA": 0.425,
    "Lateral OA": 0.12499999999999999,
    "PF OA": 0.425,
    "Effusion": 0.5249999999999999,
    "Synovitis": 0.6,
    "Baker's": 0.28750000000000003,
    "Contusion": 0.35000000000000003,
    "Fracture": 0.5499999999999999
  },
  "reader_weights": {
    "Baker's": 0.035,
    "Contusion": 0.03,
    "Medial OA": 0.025,
    "ACL": 0.025,
    "Medial Meniscus": 0.02,
    "MCL": 0.02,
    "Lateral Meniscus": 0.008,
    "Fracture": 0.008,
    "Lateral OA": 0.008,
    "PF OA": 0.008,
    "Effusion": 0.004,
    "Synovitis": 0.003
  },
  "base_float32_csv_roundtrip": true
}
''')
validate_config(CONFIG)


## Synthetic demonstration


In [ ]:
rng = np.random.default_rng(42)
labels = CONFIG['target_order']
stages = {}
for branch in STAGES:
    frame = pd.DataFrame(rng.uniform(.01, .99, (48, 12)), columns=labels)
    frame.insert(0, ID_COLUMN, [f'synthetic-{i:04d}' for i in range(48)])
    stages[branch] = frame

result = combine_stage_frames(stages, CONFIG)
assert result.shape == (48, 13)
assert np.isfinite(result[labels].to_numpy()).all()
assert result[labels].to_numpy().min() >= 0
assert result[labels].to_numpy().max() <= 1
result.to_csv('synthetic_submission.csv', index=False)
print('Synthetic demonstration passed: 48 synthetic cases, 12 findings.')
print('These outputs are not clinical predictions or a Kaggle score.')
display(result.head())


## License for this independently written mirror

MIT License

Copyright (c) 2026 kaishikan750-cyber

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
